# PCA preparation: monthly MLS N2O and O3 at the flux-measurement sites

Merges `pca_prep.ipynb` and `pca_prep_o3.ipynb` into a single pass. The output is the same `data_pca3.csv`:

1. Read the site table (`data_pca.csv`) — one row per ground flux measurement, with location, year and month.
2. For every row, gas (N2O, O3) and pressure level (68, 46, 32, 22 hPa), open the matching raw monthly MLS file and average all values for that month.
3. Store the results as `strat_<level>` (N2O) and `o3_<level>` (O3), run some sanity checks, and save.
4. Optionally, also write `data_pca4.csv`, the same data in the format the PCA notebook reads.

**Changes from the two old notebooks**
- The raw files are cleaned in memory, so the `*_cleaned` folders are no longer written or read. That also removes the risk of a stale cleaned copy being used.
- One loop covers both gases and all four levels, so no cell has to be edited and rerun per level.
- Each file is read only once (cached), even when several rows use the same location-year.
- No `os.chdir`. The output is written next to this notebook, not into a data folder.
- New: counts of values per monthly mean, a missing-data report and plausibility ranges.
- New: an optional step that writes `data_pca4.csv` directly, so the reformatting is no longer done by hand.

## 1. Settings
Paths and options. This is the only cell that should need editing.

In [ ]:
from pathlib import Path
from functools import lru_cache
import numpy as np
import pandas as pd

# --- Inputs ------------------------------------------------------------------
# Site table: one row per ground-based N2O flux measurement
SITE_TABLE = Path("data_pca.csv")

# Folders holding the RAW monthly MLS files, one sub-folder per pressure level,
# with files named monthly_<location>_<year><level>.csv  (e.g. 22/monthly_brunei_201722.csv).
# !! These are the paths from the old notebooks. Point them at the V6 extraction before rerunning. !!
N2O_DIR = Path("C:/Users/polgaste/Desktop/iasi_integration/mls_v6_test/monthly_data/data_n2o")
O3_DIR  = Path("C:/Users/polgaste/Desktop/iasi_integration/mls_v6_test/monthly_data/data_o3")

LEVELS = [68, 46, 32, 22]   # hPa

# --- Output ------------------------------------------------------------------
OUT_FILE = Path("data_pca3.csv")

# Optional second output in the format the PCA notebook reads (pca_test3: sep=";").
WRITE_PCA4 = True
PCA4_FILE = Path("data_pca4.csv")

# Minimum number of MLS values behind a monthly mean (the rule used in the trend analysis).
# False = only report months below the threshold and keep them, which reproduces the old output.
# True  = set those monthly means to NaN.
APPLY_MIN_COUNT = False
MIN_COUNT = 5

## 2. Reading one raw monthly file
Each raw file has one line per calendar month (January first), with the individual MLS values for that month separated by commas. Lines have different lengths, so `pd.read_csv` can't read them directly. Instead we split each line ourselves and build a table, which pandas pads with NaN where a line is shorter.

The table is forced to exactly 12 rows. A file with fewer lines is padded **at the end**, which assumes the missing months are the last ones of the year. A note is printed whenever this happens, so you can check that the assumption holds.

In [ ]:
def read_monthly_file(path):
    """Return a 12-row table (one row per month) of the individual values in one raw file."""
    with open(path) as f:
        rows = [line.strip().split(",") for line in f]
    df = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")   # text and blanks -> NaN
    if len(df) != 12:
        print(f"  note: {path.name} has {len(df)} lines (expected 12) - "
              f"{'padded at the end' if len(df) < 12 else 'lines after 12 ignored'}")
    return df.reindex(range(12))


@lru_cache(maxsize=None)
def load(gas_dir, level, loc, year):
    """Read (and remember) the file for one gas, level, location and year; None if it doesn't exist."""
    path = Path(gas_dir) / str(level) / f"monthly_{loc}_{year}{level}.csv"
    if not path.exists():
        print(f"  missing file: {path}")
        return None
    return read_monthly_file(path)


def monthly_mean(gas_dir, level, loc, year, month):
    """Mean and number of valid values for one location-month; (NaN, 0) if there is no data."""
    df = load(str(gas_dir), int(level), str(loc), int(year))
    if df is None:
        return np.nan, 0
    vals = df.iloc[int(month) - 1].dropna()
    return (vals.mean() if len(vals) else np.nan), len(vals)

## 3. Site table
The old V5 `68hPa` column in `data_pca.csv` is not read. All MLS columns are recomputed below.

In [ ]:
data = pd.read_csv(SITE_TABLE, usecols=["year", "month", "date_time", "Asukoht", "loc_inname",
                                         "lat", "lon", "N2O", "EDGAR"])
data

## 4. Monthly means for both gases and all levels
`counts` records how many MLS values went into each mean. It isn't saved, but step 5 uses it.

In [ ]:
counts = {}
for prefix, gas_dir in [("strat", N2O_DIR), ("o3", O3_DIR)]:
    for level in LEVELS:
        col = f"{prefix}_{level}"
        results = [monthly_mean(gas_dir, level, r.loc_inname, r.year, r.month)
                   for r in data.itertuples()]
        data[col] = [mean for mean, n in results]
        counts[col] = [n for mean, n in results]

counts = pd.DataFrame(counts, index=data.index)
value_cols = list(counts.columns)
data[["loc_inname", "year", "month"] + value_cols]

## 5. Sanity checks
- **Missing values**: a missing file, or a month with no valid values.
- **Low counts**: monthly means based on fewer than `MIN_COUNT` values. The trend analysis drops these, so decide whether the PCA should too.
- **Ranges**: a quick check for fill values (MLS uses −999.99) or unit mix-ups. Expect N2O roughly 0–350 ppbv and O3 roughly 0–10 ppmv.
- **Shared location-months**: rows that get identical MLS values because they share a location and month.

In [ ]:
missing = data[value_cols].isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "  none")

low = (counts > 0) & (counts < MIN_COUNT)
print(f"\nMonthly means based on 1-{MIN_COUNT - 1} values:")
print(low.sum()[low.sum() > 0] if low.any().any() else "  none")

print("\nValue ranges:")
print(data[value_cols].agg(["min", "max"]).round(2))

dup =data.duplicated(subset=["loc_inname", "year", "month"], keep=False)
print(f"\n{dup.sum()} of {len(data)} rows share a location-month with another row "
      f"(identical MLS values; the PCA notebook averages them into one point per location-month).")

## 6. Save
Same columns and order as the old `data_pca3.csv`.

In [ ]:
if APPLY_MIN_COUNT:
    data[value_cols] = data[value_cols].where(counts >= MIN_COUNT)

out = data.copy()
out.to_csv(OUT_FILE, index=False)
print(f"Saved {OUT_FILE.resolve()}  ({len(out)} rows, {out.shape[1]} columns)")
print(list(out.columns))

## 7. Optional: PCA input file (`data_pca4.csv`)
This writes the same data in the layout `pca_test3.ipynb` reads. The layout:
- `;` as the separator
- column names `N2O_wetland`, `N2O_inventory`, `N2O_<level>hPa`, `O3_<level>hPa`
- `Asukoht` replaced by a clean English site label, one per location

The labels are not just cosmetic. The PCA notebook groups rows by `Asukoht` to merge measurements from the same location-month, so every row of a location must carry the same label. The mapping below was taken from your existing `data_pca4.csv`. A location missing from it gets a printed warning and keeps its `Asukoht` with the trailing number removed.

This cell also works on its own. After running only the settings cell, it reads the existing `data_pca3.csv`, so you can reformat a file without redoing the extraction. It runs only when `WRITE_PCA4 = True`.

In [ ]:
SITE_LABELS = {
    "bashkortostan": "Bashkortostan",       "brunei": "Brunei",
    "california": "California",             "estonia": "Estonia",
    "florianopolis": "Santa Catarina",      "florida": "Florida",
    "khabarovsk": "Khabarovsk",             "iceland_e": "Iceland E",
    "iceland_w": "Iceland W",               "colombia": "Colombia",
    "kyrgyzstan": "Kyrgyzstan",             "morocco": "Morocco",
    "bozeman": "Montana",                   "huntingdon": "Southern Quebec",
    "mukhrino": "Mukhrino",                 "myanmar": "Myanmar",
    "pantanal": "Pantanal",                 "quistococha": "Peruvian Amazon",
    "french_guiana": "French Guiana",       "catalonia": "Catalonia",
    "france": "France",                     "romania": "Romania",
    "finland": "Finland",                   "taiwan": "Taiwan",
    "tasmania": "Tasmania",                 "tierra_del_fuego": "Tierra del Fuego",
    "uganda_e": "Uganda E",                 "uganda_n": "Uganda N",
    "uganda_s": "Uganda S",                 "nz_s": "New Zealand S",
    "nz_n": "New Zealand N",                "wales": "Wales",
}

RENAME = {"N2O": "N2O_wetland", "EDGAR": "N2O_inventory",
          **{f"strat_{lev}": f"N2O_{lev}hPa" for lev in LEVELS},
          **{f"o3_{lev}": f"O3_{lev}hPa" for lev in LEVELS}}


def to_pca4(df):
    """Rename columns and replace Asukoht with one clean label per location."""
    df = df.rename(columns=RENAME)
    unknown = sorted(set(df["loc_inname"]) - set(SITE_LABELS))
    if unknown:
        print("No label for:", unknown, "- keeping their Asukoht with the trailing number removed")
    fallback = df["Asukoht"].str.replace(r"\s*\d+$", "", regex=True)
    df["Asukoht"] = df["loc_inname"].map(SITE_LABELS).fillna(fallback)
    return df


if WRITE_PCA4:
    source = out if "out" in globals() else pd.read_csv(OUT_FILE)
    pca4 = to_pca4(source)
    pca4.to_csv(PCA4_FILE, sep=";", index=False)
    n_groups = pca4.groupby(["Asukoht", "year", "month"]).ngroups
    print(f"Saved {PCA4_FILE.resolve()}  ({len(pca4)} rows -> {n_groups} location-months in the PCA)")
    print(list(pca4.columns))